In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%pip install langchain-groq

In [0]:
%pip install -U typing-extensions>=4.13.0

In [0]:
import sys
import os

# Adds the parent directory (project root) to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from src.agent.nodes.auditor_worker import auditor_node

# ---------------------------------------------------------
# Test Case 1: Verifying Tool Invocation
# ---------------------------------------------------------
print("=== Running Test Case 1: Triggering the Database Tools ===")

# Mock the state: The user is asking for an initial audit
mock_state_1 = {
    "messages": [
        HumanMessage(content="Please investigate account C12345.")
    ],
    "next_node": "",
    "is_security_breach": False
}

# Invoke the node directly
output_1 = auditor_node(mock_state_1)
ai_message_1 = output_1["messages"][-1]

print(f"Test 1 AI Raw Output: {ai_message_1}")
print("Test 1 - Tools Requested by LLM:")
for tool in ai_message_1.tool_calls:
    print(f"- Tool Name: {tool['name']}")
    print(f"  Arguments: {tool['args']}")
# Assert that the AI didn't just talk, but actually tried to call a tool
assert hasattr(ai_message_1, 'tool_calls') and len(ai_message_1.tool_calls) > 0, "❌ Failed: Auditor did not attempt to call a tool."
assert ai_message_1.tool_calls[0]['name'] == 'get_customer_risk_profile', "❌ Failed: Auditor called the wrong tool first."

print(f"✅ Test Case 1 Passed! Auditor successfully requested execution of: {ai_message_1.tool_calls[0]['name']}\n")


# ---------------------------------------------------------
# Test Case 2: Verifying Data Analysis & Adherence to Rules
# ---------------------------------------------------------
print("=== Running Test Case 2: Analyzing Tool Data ===")

# We simulate the exact message history of a completed tool call.
# This proves the LLM can read the Unity Catalog JSON dump and explain it.
# Create two distinct mock IDs matching what a parallel tool call looks like
profile_call_id = "call_profile_123"
tx_call_id = "call_tx_123"

mock_state_2 = {
    "messages": [
        # 1. User prompt
        HumanMessage(content="Please investigate account C99999."),
        
        # 2. Simulated parallel tool request from the LLM
        AIMessage(
            content="", 
            tool_calls=[
                {"name": "get_customer_risk_profile", "args": {"account_id": "C99999"}, "id": profile_call_id, "type": "tool_call"},
                {"name": "get_recent_transactions", "args": {"account_id": "C99999", "limit": 10}, "id": tx_call_id, "type": "tool_call"}
            ]
        ),
        
        # 3. Simulated response from the Risk Profile Tool (showing a severe ledger discrepancy)
        ToolMessage(
            content='[{"account_id": "C99999", "total_transactions": 5, "total_volume_moved": 10000.0, "avg_tx_size": 2000.0, "high_risk_tx_count": 2, "total_discrepancy_amount": 500.0, "has_prior_system_flags": 1}]',
            tool_call_id=profile_call_id
        ),
        
        # 4. Simulated response from the Recent Transactions Tool
        ToolMessage(
            content='[{"tx_id": "TX_001", "amount": 5000.0, "type": "TRANSFER", "is_fraud": 0}]',
            tool_call_id=tx_call_id
        )
    ],
    "next_node": "",
    "is_security_breach": False
}

# Invoke the auditor node with the complete data package
output_2 = auditor_node(mock_state_2)
final_analysis = output_2["messages"][-1].content

print(f"Test 2 Final Analysis Generated by Groq:\n\n{final_analysis}\n")

# Assert that the LLM recognized the discrepancy and flagged it
has_discrepancy_flag = "SEVERE" in final_analysis.upper() or "discrepancy" in final_analysis.lower()
assert has_discrepancy_flag, "❌ Failed: Auditor missed the ledger discrepancy rule."

print("✅ Test Case 2 Passed! Auditor correctly parsed the parallel JSON feeds and generated an explainable compliance summary.")